In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder


df = pd.read_csv('/content/MyDrive/ML/data(11).csv', encoding='ISO-8859-1')

print(df.head())
print("Dataset shape:", df.shape)

df = df.dropna(subset=['InvoiceNo', 'Description'])

df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

df = df[df['Quantity'] > 0]

transactions = df.groupby('InvoiceNo')['Description'].apply(
    lambda items: list(set(items))
).tolist()

transactions = [
    items for items in transactions if len(items) >= 2
]

transactions = transactions[:5000]

print("Number of transactions:", len(transactions))

encoder = TransactionEncoder()

encoded_array = encoder.fit(transactions).transform(transactions)

basket = pd.DataFrame(
    encoded_array,
    columns=encoder.columns_
)

print("\nEncoded transaction data:")
print(basket.head())

frequent_itemsets = apriori(
    basket,
    min_support=0.02,
    use_colnames=True
)

print("\nFrequent itemsets:")
print(frequent_itemsets.head(10))

if not frequent_itemsets.empty:
    rules = association_rules(
        frequent_itemsets,
        metric='confidence',
        min_threshold=0.2
    )

    if not rules.empty:
        rules = rules.sort_values(
            by='lift',
            ascending=False
        )

        print("\nTop association rules:")
        print(
            rules[
                ['antecedents', 'consequents',
                 'support', 'confidence', 'lift']
            ].head(10).to_string(index=False)
        )

        plt.scatter(
            rules['support'],
            rules['confidence']
        )

        plt.xlabel("Support")
        plt.ylabel("Confidence")
        plt.title("Association Rules")
        plt.show()
    else:
        print("No association rules found. Try a lower threshold.")
else:
    print("No frequent itemsets found. Try a lower support threshold.")